In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import torch
import copy
from torch.utils.data import DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from tqdm import tqdm

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from src.trainer import SimpleTrainer, SITrainer, IntervalTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils import InContextHead
from src import models
from src.buffer import MultiTaskBuffer
from src.interval_utils import create_violation_mask
from src.regulariser import MultiRegulariser, L2Regulariser, UnbiasRegulariser

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()
model.to("mps")
model.train()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
train_tasks, buffer_sets = zip(
    *[create_holdout_set(dataset) for dataset in train_tasks]
)
print([len(task) for task in buffer_sets])

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:2])

In [ ]:
save_model = copy.deepcopy(trainer.model)

In [ ]:
interval_trainer = IntervalTrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

# Train task 1 until plateau
interval_trainer.train(train_tasks[1], val_tasks[1], batch_size=256, epochs=3)
interval_trainer.test(test_tasks[0:2])

In [ ]:
si_trainer = SITrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

# Compute bounds for task 0
si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0)

# Train task 1 until plateau
si_trainer.test(test_tasks[0:2])
si_trainer.train(train_tasks[1], val_tasks[1], batch_size=256, epochs=3)
si_trainer.test(test_tasks[0:2])

In [ ]:
si_trainer = SITrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=1,
    shuffle=True,
    generator=torch.Generator().manual_seed(0),
)
optimizer = torch.optim.Adam(si_trainer.model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

si_trainer.si.activate()
si_trainer.model.train()
for _ in range(10):
    inputs, targets = next(iter(loader))
    loss, _ = si_trainer._train_step(
        si_trainer.model,
        inputs,
        targets,
        optimizer,
        loss_fn,
        project=False,
    )
    si_trainer.si.update_importance()
si_trainer.si.deactivate()

temp_model = copy.deepcopy(save_model)
si_trainer.model = temp_model
si_trainer.si.model = temp_model
si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0.7)
si_trainer.test(test_tasks[0:2])
si_trainer.train(train_tasks[1], val_tasks[1], epochs=3, batch_size=256)
si_trainer.test(test_tasks[0:2])

In [ ]:
def run_experiment(seed, lookahead_steps=10, require_prev=True, pruning_prop=0.7):
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)

    model = models.get_mnist_model(seed=seed)
    model.to("mps")

    train_tasks, _ = zip(*[create_holdout_set(dataset) for dataset in train_tasks])
    trainer = SimpleTrainer(model, seed=seed)
    trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
    trainer.test(test_tasks[0:1])

    save_model = copy.deepcopy(trainer.model)

    interval_trainer = IntervalTrainer(
        save_model,
        checkpoint=200,
        n_iters=200,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    # Compute bounds for task 0
    prev = []
    if require_prev:
        interval_trainer.compute_rashomon_set(test_tasks[0])

        # Train task 1 until plateau
        interval_trainer.train(train_tasks[1], val_tasks[1], batch_size=256, epochs=3)
        print("No pruning performance: ", end="")
        prev = interval_trainer.test(test_tasks[0:2])

    si_trainer = SITrainer(
        save_model,
        checkpoint=200,
        n_iters=200,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    loader = DataLoader(
        train_tasks[1],
        batch_size=32,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    optimizer = torch.optim.Adam(si_trainer.model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    si_trainer.si.activate()
    si_trainer.model.train()
    for _ in range(lookahead_steps):
        inputs, targets = next(iter(loader))
        loss, _ = si_trainer._train_step(
            si_trainer.model,
            inputs,
            targets,
            optimizer,
            loss_fn,
            project=False,
        )
        si_trainer.si.update_importance()
    si_trainer.si.deactivate()

    si_trainer.model = save_model
    si_trainer.si.model = save_model
    si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=pruning_prop)
    si_trainer.train(train_tasks[1], val_tasks[1], epochs=3, batch_size=256)
    new = si_trainer.test(test_tasks[0:2])

    return prev, new

In [ ]:
prev_perf = []
new_perf = []
# reds = []
for i in range(10):
    result = run_experiment(i, lookahead_steps=50, pruning_prop=0.8)
    prev_perf.append(result[0])
    new_perf.append(result[1])
    # reds.append(result[2])

In [ ]:
prev_acc = np.array([val[1][1] for val in prev_perf]).mean()
prev_std = np.array([val[1][1] for val in prev_perf]).std()
new_acc = np.array([val[1][1] for val in new_perf]).mean()
new_std = np.array([val[1][1] for val in new_perf]).std()

# 2. Create a prettier plot
# Use a more balanced figure size
fig, ax = plt.subplots(figsize=(7, 6))

# Define some nice colors
colors = ["#4682B4", "#5FBA7D"]  # Steel Blue and a nice Green

# Plot the bars with improved styling
bars = ax.bar(
    x=["Previous", "New"],
    height=[prev_acc, new_acc],
    yerr=[prev_std, new_std],
    color=colors,
    alpha=0.8,  # Make bars slightly transparent
    edgecolor="black",  # Add a crisp black edge
    capsize=10,  # THIS IS KEY: Adds caps to the error bars
    ecolor="black",  # Color of the error bar lines
    linewidth=1.5,
)

# 3. Add Labels, Title, and Grid for context
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Comparison of Previous vs. New Performance", fontsize=16, pad=20)
ax.set_xticks(ticks=[0, 1], labels=["No lookahead", "50 step lookahead"], fontsize=12)

# Add a subtle horizontal grid to make comparisons easier
ax.yaxis.grid(True, linestyle="--", which="major", color="grey", alpha=0.3)

# Remove the top and right spines for a cleaner look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Set a dynamic Y-axis limit for better spacing
ax.set_ylim(0, max(new_acc, prev_acc) + max(new_std, prev_std) + 0.05)

# 4. Add data labels on top of the bars
# This shows the exact mean value on the plot
ax.bar_label(bars, fmt="{:.3f}", padding=5, fontsize=11, color="black")

# Ensure everything fits without overlapping
plt.tight_layout()
plt.show()

In [ ]:
data = defaultdict(list)
pruning_prop = [0, 0.05, 0.1, 0.3, 0.5, 0.7, 0.8, 0.85]
for prop in tqdm(pruning_prop):
    for i in range(3):
        data[prop].append(run_experiment(i, pruning_prop=prop, lookahead_steps=20))

In [ ]:
x_vals, y_means, y_stds = [], [], []
prev_means = []
prop_mean = []
for key, item in data.items():
    x_vals.append(key)
    # Create the numpy array once
    values = np.array([val[1][1][1] for val in item])
    y_means.append(values.mean())
    y_stds.append(values.std())
    prop_mean.append(np.array([val[0] for val in item]).mean())
    prev_means.append(np.array([val[0][1][1] for val in item]).mean())

# Convert lists to numpy arrays for easier calculations
x_vals = np.array(x_vals)
y_means = np.array(y_means)
y_stds = np.array(y_stds)

# 2. Create a prettier plot
# Use a pre-defined style for a professional look
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

# Define a color for the plot
primary_color, secondary_color = ["#4682B4", "#5FBA7D"]

# 3. Plot the main line with markers
ax.plot(
    x_vals,
    y_means,
    color=primary_color,
    linewidth=2.5,
    marker="o",  # Add circles at each data point
    markersize=8,
    label="Mean Performance",
)

ax.plot(
    x_vals,
    prev_means,
    color=secondary_color,
    linewidth=2.5,
    markersize=8,
    linestyle="--",
    label="Mean Baseline Performance",
)

# 4. Add the shaded error band (the key improvement!)
# The alpha value makes the shade transparent
ax.fill_between(
    x_vals,
    y_means - y_stds,  # Lower bound of the error
    y_means + y_stds,  # Upper bound of the error
    color=primary_color,
    alpha=0.2,
    label="Standard Deviation",
)

# 5. Add Labels, Title, and Legend
ax.set_title("Performance vs. Pruning Level", fontsize=16, pad=20)
ax.set_xlabel("Pruning Proportion", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.legend(loc="upper right", fontsize=11)

# Customize tick parameters
ax.tick_params(axis="both", which="major", labelsize=11)

# plt.plot(x_vals, prop_mean)

# Ensure everything fits nicely
plt.tight_layout()
plt.show()


In [ ]:
data2 = defaultdict(list)
lookaheads = [0, 1, 5, 10, 20, 50, 100, 200]
for lookahead in tqdm(lookaheads):
    for i in range(5):
        data2[lookahead].append(
            run_experiment(
                i, pruning_prop=0.8, require_prev=False, lookahead_steps=lookahead
            )
        )

In [ ]:
x_vals, y_means, y_stds = [], [], []
prop_mean = []
for key, item in data2.items():
    x_vals.append(key)
    # Create the numpy array once
    values = np.array([val[1][1][1] for val in item])
    y_means.append(values.mean())
    y_stds.append(values.std())
    prop_mean.append(np.array([val[0] for val in item]).mean())

# Convert lists to numpy arrays for easier calculations
x_vals = np.array(x_vals)
y_means = np.array(y_means)
y_stds = np.array(y_stds)

# 2. Create a prettier plot
# Use a pre-defined style for a professional look
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

# Define a color for the plot
primary_color = "#007ACC"  # A nice, clear blue

# 3. Plot the main line with markers
ax.plot(
    x_vals,
    y_means,
    color=primary_color,
    linewidth=2.5,
    marker="o",  # Add circles at each data point
    markersize=8,
    label="Mean Performance",
)

# 4. Add the shaded error band (the key improvement!)
# The alpha value makes the shade transparent
ax.fill_between(
    x_vals,
    y_means - y_stds,  # Lower bound of the error
    y_means + y_stds,  # Upper bound of the error
    color=primary_color,
    alpha=0.2,
    label="Standard Deviation",
)

# 5. Add Labels, Title, and Legend
ax.set_title("Performance vs. Lookahead", fontsize=16, pad=20)
ax.set_xlabel("Lookahead steps", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.legend(loc="upper right", fontsize=11)

# Customize tick parameters
ax.tick_params(axis="both", which="major", labelsize=11)

# plt.plot(x_vals, prop_mean)

# Ensure everything fits nicely
plt.tight_layout()
plt.show()


In [ ]:
si_trainer = SITrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=1,
    shuffle=True,
    generator=torch.Generator().manual_seed(0),
)
optimizer = torch.optim.Adam(si_trainer.model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

si_trainer.si.activate()
si_trainer.model.train()
for _ in range(10):
    inputs, targets = next(iter(loader))
    loss, _ = si_trainer._train_step(
        si_trainer.model,
        inputs,
        targets,
        optimizer,
        loss_fn,
        project=False,
    )
    si_trainer.si.update_importance()
si_trainer.si.deactivate()

temp_model = copy.deepcopy(save_model)
si_trainer.model = temp_model
si_trainer.si.model = temp_model
si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0.7)
si_trainer.test(test_tasks[0:2])
si_trainer.train(train_tasks[1], val_tasks[1], epochs=3, batch_size=256)
si_trainer.test(test_tasks[0:2])

In [ ]:
# HYPERPARAMS
# checkpoint
# n_iters
# primal_learning_rate
# dual_learning_rate
# batch_size
# learning rate
# pruning proportion
# penalty coefficient


def tuning_func(
    checkpoint: int = 100,
    n_iters: int = 300,
    min_acc_limit: float = 0.8,
    min_acc_increment: float = 0.05,
    primal_learning_rate: float = 0.5,
    dual_learning_rate: float = 0.1,
    learning_rate: float = 0.001,
    weight_decay: float = 0.01,
    l2_lambda: float = 0.01,
    unbias_lambda: float = 0.01,
    device="cuda",
):
    SEED = 0
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

    model = models.get_mnist_model(seed=SEED)
    model.to(device)

    l2 = L2Regulariser(lmbd=l2_lambda)
    unbias = UnbiasRegulariser(lmbd=unbias_lambda)
    regulariser = MultiRegulariser([unbias, l2])

    train_tasks, _ = zip(*[create_holdout_set(dataset) for dataset in train_tasks])
    trainer = SimpleTrainer(model, seed=SEED)
    trainer.train(
        train_tasks[0], val_tasks[0], epochs=3, batch_size=256, regulariser=regulariser
    )
    trainer.test(test_tasks[0:1])

    save_model = copy.deepcopy(trainer.model)

    si_trainer = SITrainer(
        save_model,
        checkpoint=checkpoint,
        n_iters=n_iters,
        min_acc_limit=min_acc_limit,
        min_acc_increment=min_acc_increment,
        primal_learning_rate=primal_learning_rate,
        dual_learning_rate=dual_learning_rate,
        paradigm="CIL",
        seed=SEED,
    )

    for i, (train, val, test) in enumerate(
        zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
    ):
        loader = DataLoader(
            train,
            batch_size=32,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED),
        )

        optimizer = torch.optim.Adam(si_trainer.model.parameters(), lr=0.001)
        loss_fn = nn.CrossEntropyLoss()

        temp_model = copy.deepcopy(si_trainer.model)

        si_trainer.si.activate()
        si_trainer.model.train()
        si_trainer.si.reset()
        for _ in range(50):
            inputs, targets = next(iter(loader))
            loss, _ = si_trainer._train_step(
                si_trainer.model,
                inputs,
                targets,
                optimizer,
                loss_fn,
                project=False,
                regulariser=regulariser,
            )
            si_trainer.si.update_importance()
        si_trainer.si.deactivate()

        si_trainer.model = temp_model
        si_trainer.si.model = temp_model
        si_trainer.compute_rashomon_set(test_tasks[i - 1], prune_prop=0)
        assert si_trainer.test(test_tasks[0 : i + 1])[-1][1] == 0, (
            "Prior last task performance needs to be zero."
        )
        si_trainer.train(
            train,
            val,
            epochs=20,
            batch_size=256,
            early_stopping=True,
            patience=50,
            lr=learning_rate,
            weigth_decay=weight_decay,
            regulariser=regulariser,
        )
        results = si_trainer.test(test_tasks[0 : i + 1])

        if results[-1][1] == 0:
            print("Catastrophic Forgetting occurred.")
            break

    return results

In [1]:
# It's always a good idea to have the latest version
!pip install wandb --upgrade
!pip install nbformat

import wandb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import copy

# Log in to your W&B account
wandb.login()

wandb: Currently logged in as: leo_elm to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
sweep_config = {
    'method': 'bayes',  # Use Bayesian Optimization
    'metric': {
        'name': 'final_avg_accuracy',
        'goal': 'maximize'
    },
    'parameters': {
        'checkpoint': {
            'values': [20, 50, 100, 150]
        },
        'n_iters': {
            'distribution': 'q_uniform',
            'min': 200,
            'max': 500,
            'q': 50
        },
        'min_acc_limit': {
            'distribution': 'uniform',
            'min': 0.7,
            'max': 0.95
        },
        'min_acc_increment': {
            'distribution': 'uniform',
            'min': 0.01,
            'max': 0.1
        },
        'primal_learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 0.01,
            'max': 1.0
        },
        'dual_learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 0.01,
            'max': 1.0
        },
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 0.0001,
            'max': 0.01
        },
        'weight_decay': {
            'distribution': 'log_uniform_values',
            'min': 1.0e-5,
            'max': 1.0e-2
        },
        'l2_lambda': {
            'distribution': 'log_uniform_values',
            'min': 1.0e-4,
            'max': 1.0e-1
        },
        'unbias_lambda': {
            'distribution': 'log_uniform_values',
            'min': 1.0e-4,
            'max': 1.0e-1
        }
    }
}

In [ ]:
def train_one_run():
    # This function will be called by the W&B agent for each run.
    
    # Initialize a W&B run. W&B automatically fills wandb.config
    # with the hyperparameters for this run.
    with wandb.init() as run:
        config = wandb.config

        # Call your main logic with the hyperparameters
        final_results = tuning_func(
            checkpoint=config.checkpoint,
            n_iters=config.n_iters,
            min_acc_limit=config.min_acc_limit,
            min_acc_increment=config.min_acc_increment,
            primal_learning_rate=config.primal_learning_rate,
            dual_learning_rate=config.dual_learning_rate,
            learning_rate=config.learning_rate,
            weight_decay=config.weight_decay,
            l2_lambda=config.l2_lambda,
            unbias_lambda=config.unbias_lambda,
            device='mps'
        )

        # Process and log the final metrics
        if final_results:
            accuracies = [res[1] for res in final_results]
            avg_accuracy = np.mean(accuracies)
            
            wandb.log({
                "final_avg_accuracy": avg_accuracy,
                "second_task_accuracy": accuracies[1] if len(accuracies) > 1 else 0
            })
        else:
            # Log a failure if catastrophic forgetting happened early
            wandb.log({"final_avg_accuracy": 0.0})

In [ ]:
# 1. Initialize the sweep
# Replace "your-project-name" with a name for your W&B project
sweep_id = wandb.sweep(sweep_config, project="class_incremental-sweep")

# 2. Run the agent. This will execute your `train_one_run` function `count` times.
# This cell will run for a long time as it trains the model for each trial.
wandb.agent(sweep_id, function=train_one_run, count=15)